***

* [总目录](../0_Introduction/0_introduction.ipynb)
* [术语表](../0_Introduction/1_glossary.ipynb)
* [第 2 章：干涉测量的数学工具箱](2_0_introduction.ipynb)
    * 上一节：[2.7 傅里叶定理：移位、缩放与卷积的统一语言](2_7_fourier_theorems.ipynb)
    * 下一节：[2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)

***


导入标准模块。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
try:
    from IPython.display import HTML
except ImportError:
    def HTML(*args, **kwargs):
        return None
%matplotlib inline
HTML('../style/course.css')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12


## 2.8 离散傅里叶变换与 FFT：从解析公式到可计算算法<a id='math:sec:the_discrete_fourier_transform_and_the_fast_fourier_transform'></a>

连续傅里叶变换作用在连续函数上；实际数据处理面对的却是有限长度的数字数组。相关器输出有限个时间积分和频率通道，图像网格包含有限个像素，`uv` 平面在计算机中也必须落到有限网格上。因此，真正被程序反复调用的不是连续傅里叶积分本身，而是离散傅里叶变换（DFT）及其快速算法（FFT）。

DFT 不是一种新的物理变换，而是连续傅里叶思想在“有限、离散、周期”模型下的精确代数版本。理解这一点非常重要：DFT 输入的是一个长度为 $N$ 的序列，输出的是 $N$ 个离散频率系数；它默认这个序列可以周期延拓，也默认频域索引同样按模 $N$ 周期重复。许多数值现象，例如负频率排列、循环卷积、边界回绕、谱泄漏和零填充，都来自这个有限周期模型。


### 2.8.1 三个层次：连续、无限离散、有限离散<a id='math:sec:ctft_dtft_dft'></a>

傅里叶语言进入计算机时，常遇到三个相近但不同的对象。连续傅里叶变换（CTFT）处理连续函数 $f(t)$，频率变量也是连续的。离散时间傅里叶变换（DTFT）处理定义在所有整数索引上的无限序列 $y_n$，频率变量连续但周期重复。离散傅里叶变换（DFT）处理长度为 $N$ 的有限序列，频域也只剩 $N$ 个离散频率系数。

DTFT 可写为

<a id='math:eq:dtft_definition'></a><!--\label{math:eq:dtft_definition}-->
$$
Y(\omega)=\sum_{n=-\infty}^{+\infty}y_n e^{-i\omega n}.
$$

由于 $e^{-i(\omega+2\pi)n}=e^{-i\omega n}$，$Y(\omega)$ 对 $\omega$ 具有 $2\pi$ 周期。若序列来自连续信号的均匀采样，$t_n=n\Delta t$，采样频率 $f_s=1/\Delta t$，则可写成

$$
Y(f)=\sum_{n=-\infty}^{+\infty}y_n e^{-2\pi i f n\Delta t},
$$

其频谱以 $f_s$ 为周期重复。第 2.9 节会进一步说明，这种周期重复正是采样导致混叠的根源。


泊松求和公式把采样与频谱复制联系起来。若连续信号 $y(t)$ 的傅里叶变换为 $Y(f)$，则在适当条件下有

<a id='math:eq:poisson_summation_sampling'></a><!--\label{math:eq:poisson_summation_sampling}-->
$$
\sum_{n=-\infty}^{+\infty}\Delta t\,y(n\Delta t)e^{-2\pi i f n\Delta t}
=\sum_{k=-\infty}^{+\infty}Y(f-kf_s).
$$

左边从离散样本出发，右边显示连续频谱按采样频率周期复制。这个公式先不必作为分析定理深究；在本章语境中，它的意义是：离散采样并不会只保留一个孤立频谱，而是把频谱副本排成一列。若副本重叠，就产生混叠。


### 2.8.2 DFT 的定义与频率网格<a id='math:sec:the_discrete_fourier_transform_definition'></a>

令

$$
y=(y_0,y_1,\ldots,y_{N-1})
$$

是长度为 $N$ 的复序列。DFT 定义为

<a id='math:eq:dft_definition'></a><!--\label{math:eq:dft_definition}-->
$$
Y_k=\sum_{n=0}^{N-1}y_n e^{-2\pi i nk/N},
\qquad k=0,1,\ldots,N-1.
$$

逆 DFT 为

<a id='math:eq:idft_definition'></a><!--\label{math:eq:idft_definition}-->
$$
y_n=\frac{1}{N}\sum_{k=0}^{N-1}Y_k e^{+2\pi i nk/N},
\qquad n=0,1,\ldots,N-1.
$$

这里把归一化因子 $1/N$ 放在逆变换中。NumPy 的 `fft`/`ifft` 默认也采用这一约定；有些教材会采用对称归一化 $1/\sqrt{N}$，或者把归一化放到正变换中。归一化约定不同只会改变系数尺度，不会改变频率位置和相位关系。


若样本间隔为 $\Delta t$，总观测长度为

$$
T=N\Delta t,
$$

DFT 的频率间隔为

<a id='math:eq:dft_frequency_spacing'></a><!--\label{math:eq:dft_frequency_spacing}-->
$$
\Delta f=\frac{1}{T}=\frac{1}{N\Delta t}.
$$

第 $k$ 个频率通道对应

$$
f_k=\frac{k}{N\Delta t},
$$

但这个表达只按 DFT 原始索引排列。由于频谱按采样频率 $f_s=1/\Delta t$ 周期重复，通常会把高于 Nyquist 频率的一半解释为负频率。常见的 `fftshift` 操作只是重新排列数组，使负频率在左、零频率在中间、正频率在右；它不改变 DFT 本身。

![DFT 的周期模型与频率 bin](figures/dft_periodic_sequence_bins.png)

**图 2.8.1** DFT 的有限周期模型。长度为 $N$ 的输入序列被视作一个周期；输出也是 $N$ 个周期重复的频率系数。使用 `fftshift` 后，负频率和正频率按更接近连续傅里叶变换的方式排列。


### 2.8.3 离散正交性与逆变换为什么成立<a id='math:sec:dft_orthogonality'></a>

DFT 的本质，是把长度为 $N$ 的向量投影到 $N$ 个离散复指数基底上。定义

$$
\phi_k(n)=e^{2\pi i nk/N}.
$$

这些基底满足离散正交性：

<a id='math:eq:dft_basis_orthogonality'></a><!--\label{math:eq:dft_basis_orthogonality}-->
$$
\sum_{n=0}^{N-1}e^{2\pi i n(k-k')/N}
=N\delta_{kk'}.
$$

当 $k=k'$ 时，每一项都是 1，求和为 $N$；当 $k\ne k'$ 时，这是一组单位圆上等间隔相量的几何级数，总和为 0。这个性质保证了正变换分析出来的系数，能够通过逆变换准确合成回原始序列。

把 DFT 写成矩阵形式更直观。令

$$
W_{kn}=e^{-2\pi i kn/N},
$$

则

$$
\mathbf{Y}=\mathbf{W}\mathbf{y},
\qquad
\mathbf{y}=\frac{1}{N}\mathbf{W}^*\mathbf{Y}.
$$

因为 $\mathbf{W}^*\mathbf{W}=N\mathbf{I}$，所以逆变换确实恢复原向量。第 2.10 节会用更系统的线性代数语言重新审视这种矩阵观点。


DFT 还具有两个直接后果。第一，索引按模 $N$ 周期重复：

$$
Y_{k+N}=Y_k,
\qquad y_{n+N}=y_n.
$$

第二，若输入序列为实数，则频谱具有 Hermitian 对称性：

<a id='math:eq:dft_hermitian_symmetry'></a><!--\label{math:eq:dft_hermitian_symmetry}-->
$$
Y_{N-k}=Y_k^*.
$$

因此，实值输入的负频率部分并不包含新的独立信息。实际软件常利用这一点提供 `rfft` 一类只计算非冗余半谱的算法。


### 2.8.4 循环卷积、边界回绕与零填充<a id='math:sec:circular_convolution'></a>

长度为 $N$ 的 DFT 默认所有序列都按周期 $N$ 延拓。因此，DFT 框架下自然出现的是循环卷积，而不是无限长线性卷积。对两个长度为 $N$ 的序列，循环卷积定义为

<a id='math:eq:circular_convolution'></a><!--\label{math:eq:circular_convolution}-->
$$
(y\circledast z)_k=\sum_{n=0}^{N-1}y_n z_{(k-n)\,\mathrm{mod}\,N}.
$$

相应的离散卷积定理为

<a id='math:eq:dft_convolution_theorem'></a><!--\label{math:eq:dft_convolution_theorem}-->
$$
\mathrm{DFT}\{y\circledast z\}=Y_k Z_k.
$$

反过来，逐点相乘对应循环卷积：

$$
\mathrm{DFT}\{y_n z_n\}=\frac{1}{N}(Y\circledast Z)_k.
$$

这里的 $1/N$ 来自本节采用的归一化约定。


循环卷积最容易在实践中被误用。若普通线性卷积的结果超出数组右端，在循环卷积中它会从左端绕回；这会把本不相邻的结构混在一起。为了用 FFT 计算普通线性卷积，必须把序列零填充到足够长。若两个序列长度分别为 $N$ 和 $M$，线性卷积长度为 $N+M-1$，因此 FFT 长度至少要达到这个值。

![循环卷积与零填充](figures/dft_circular_convolution_padding.png)

**图 2.8.2** DFT 乘法对应循环卷积。没有零填充时，边界外的响应会绕回数组开头；足够零填充后，FFT 卷积与普通线性卷积一致。


在干涉成像中，这一点会以多种形式出现。FFT 图像天然具有周期边界；gridding 卷积核有有限支撑；图像边缘若没有足够留白，旁瓣或卷积尾部可能从另一侧回绕。实际软件中的 padding、oversampling 和裁切策略，都是为了控制这些周期边界效应。


### 2.8.5 DFT 的直接实现与计算代价<a id='math:sec:numerical_DFT'></a>

按照定义直接计算 DFT，需要对每个 $k$ 都把 $N$ 个样本求和。共有 $N$ 个输出频率，因此计算量随 $N^2$ 增长：

$$
Y_k=\sum_{n=0}^{N-1}y_n e^{-2\pi i nk/N}.
$$

矩阵乘法形式

$$
\mathbf{Y}=\mathbf{W}\mathbf{y}
$$

可以利用底层线性代数库加速，但本质上仍然是 $N\times N$ 矩阵乘以向量，复杂度仍为 $\mathcal{O}(N^2)$。当 $N$ 很大时，这种方法很快变得不可接受。一个 $4096\times4096$ 图像的二维 DFT 若按朴素方式计算，代价会大到难以作为日常成像步骤使用。


### 2.8.6 FFT：更快地计算同一个 DFT<a id='math:sec:fast_fourier_transforms'></a>

快速傅里叶变换（FFT）不是新的变换，而是计算 DFT 的快速算法。最经典的 Cooley-Tukey 思想，是把长度为 $N$ 的 DFT 拆成偶数索引和奇数索引两部分。令 $N$ 为偶数，

$$
Y_k=\sum_{n=0}^{N-1}y_n e^{-2\pi i nk/N}.
$$

把 $n=2r$ 与 $n=2r+1$ 分开：

<a id='math:eq:cooley_tukey_split'></a><!--\label{math:eq:cooley_tukey_split}-->
$$
\begin{aligned}
Y_k
&=\sum_{r=0}^{N/2-1}y_{2r}e^{-2\pi i(2r)k/N}
+\sum_{r=0}^{N/2-1}y_{2r+1}e^{-2\pi i(2r+1)k/N}\\
&=E_k+e^{-2\pi ik/N}O_k,
\end{aligned}
$$

其中 $E_k$ 与 $O_k$ 是偶数子序列和奇数子序列的长度 $N/2$ DFT。再利用周期性，可得

$$
Y_{k+N/2}=E_k-e^{-2\pi ik/N}O_k.
$$

这样，一个长度 $N$ 的问题被拆成两个长度 $N/2$ 的问题，加上一组称为 twiddle factors 的相位因子。递归重复这一过程，复杂度从 $\mathcal{O}(N^2)$ 降到 $\mathcal{O}(N\log N)$。

![DFT 与 FFT 的复杂度比较](figures/dft_fft_complexity.png)

**图 2.8.3** 直接 DFT 与 FFT 的复杂度增长。FFT 的数学结果仍是 DFT，但可计算规模发生了根本变化。


现代 FFT 库还会使用混合基数分解、实序列优化、缓存友好的内存布局、多线程和硬件指令优化。对本书而言，最重要的结论是：FFT 让大规模傅里叶计算成为日常工具。干涉成像中的快速成图、卷积、PSF 计算和谱分析，都依赖这一算法层面的突破。


### 2.8.7 与干涉成像的关系<a id='math:sec:dft_fft_interferometry'></a>

干涉阵测量的可见度并不天然落在规则笛卡尔网格上，而 FFT 要求规则网格数据。因此实际成像通常包含一个关键中间步骤：把不规则 `uv` 样本通过 gridding 卷积到规则网格上，再用 FFT 从 `uv` 平面变换到图像平面。这个过程会引入卷积核、权重、零填充、裁切和归一化等数值选择。

DFT 和 FFT 在这里的角色可以这样理解：DFT 给出有限离散傅里叶对的精确定义，FFT 让这种变换可以快速计算，而 gridding 则负责把真实干涉数据变成 FFT 可以处理的规则数组。第 2.9 节将继续讨论采样、分辨率和混叠；后续成像章节会把这些数学概念放回真实的 `uv` 覆盖与脏波束中。


***

* 下一节：[2.9 采样理论：离散测量的分辨率与混叠](2_9_sampling_theory.ipynb)
